# Assignment 6 — Image Restoration

Implements log-domain restoration, median restoration, and contraharmonic filtering.

This notebook is intentionally modular. **Style 1** uses OpenCV operations; **Style 2** uses OpenCV only for reading and Matplotlib for display, while the image-processing logic is written with arrays, loops, queues, and formulas.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

LAB_DIR = Path.cwd()

def show_images(images, titles, cmap="gray", figsize=(14, 4)):
    """Display a sequence of images consistently in a notebook."""
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    axes = np.atleast_1d(axes)
    for axis, image, title in zip(axes, images, titles):
        axis.imshow(image, cmap=cmap)
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()


## Style 1 — OpenCV built-in operations

In [ ]:
def restore_cv(image):
    log_image = np.log1p(image.astype(np.float32))
    smooth = cv2.blur(log_image, (3, 3))
    return np.clip(np.expm1(smooth), 0, 255).astype(np.uint8)

def median_restore_cv(image):
    return cv2.medianBlur(image, 3)

def contraharmonic_cv(image, q, size=5):
    numerator = cv2.boxFilter(image.astype(np.float32) ** (q + 1), -1, (size, size))
    denominator = cv2.boxFilter(np.maximum(image, 1).astype(np.float32) ** q, -1, (size, size))
    return np.clip(numerator / np.maximum(denominator, 1e-6), 0, 255).astype(np.uint8)


## Style 2 — DSA/manual implementation

In [ ]:
def mean_manual(image):
    padded = np.pad(image.astype(float), 1, mode="edge"); result = np.zeros_like(image, float)
    for r in range(image.shape[0]):
        for c in range(image.shape[1]):
            result[r, c] = padded[r:r + 3, c:c + 3].mean()
    return result

def median_manual(image):
    padded = np.pad(image, 1, mode="edge"); result = np.zeros_like(image)
    for r in range(image.shape[0]):
        for c in range(image.shape[1]):
            result[r, c] = sorted(padded[r:r + 3, c:c + 3].ravel())[4]
    return result

def contraharmonic_manual(image, q, size=5):
    pad = size // 2; padded = np.pad(image.astype(float), pad, mode="edge"); result = np.zeros_like(image, float)
    for r in range(image.shape[0]):
        for c in range(image.shape[1]):
            w = np.maximum(padded[r:r + size, c:c + size], 1e-6)
            result[r, c] = (w ** (q + 1)).sum() / (w ** q).sum()
    return np.clip(result, 0, 255).astype(np.uint8)


## Demonstration

In [ ]:
image = cv2.imread(str(LAB_DIR / "Blurry_moon.tif"), cv2.IMREAD_GRAYSCALE)
if image is None: image = np.tile(np.arange(256, dtype=np.uint8), (256, 1))
noisy = image.copy(); noisy[::11, ::13] = 0; noisy[5::17, 7::19] = 255
restored_cv = median_restore_cv(noisy); restored_manual = median_manual(noisy)
print("Median output equality:", np.array_equal(restored_cv, restored_manual))
show_images([image, noisy, restored_cv, restored_manual], ["Original", "Salt-pepper sample", "cv2 median", "Manual median"], figsize=(16, 4))
